In [1]:
#### ------------------------------------------------------------------------------------------
#### author: Ranjan Barman, date: Mar 6, 2025
#### Compute NPIFs based on HoverNet prediction, using MPP = 0.248 for unit conversion
#### Removes outliers for Major Axis and Minor Axis using IQR
#### Filters top X% tiles based on majority of cancer nuclei
#### ------------------------------------------------------------------------------------------

import os
import pandas as pd
import numpy as np
import argparse

# Parse command-line arguments
parser = argparse.ArgumentParser()
parser.add_argument("--percentile", type=int, required=True, help="Top X percentile for filtering tiles")
args = parser.parse_args()

# Set working directory
_wpath_ = "/data/Lab_ruppin/Ranjan/HnE/"
os.makedirs(_wpath_, exist_ok=True)
os.chdir(_wpath_)

print(f"Working directory: {_wpath_}")

# Define dataset paths
dataset_name = "TCGA_BRCA_FFPE"
input_folder = f"{dataset_name}/outputs/HoverNet/"
output_file_path = f"{dataset_name}/outputs/HoverNet/HoverNet_NPIFs_TCGA_BRCA_1106_Filtered_Top{args.percentile}Q.csv"

# Define computation settings
columns_to_compute = ["Area", "Major Axis", "Minor Axis", "Perimeter", "Eccentricity", "Circularity"]
MPP = 0.248  

def remove_outliers(df, column):
    Q1, Q3 = df[column].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    return df[(df[column] >= Q1 - 3 * IQR) & (df[column] <= Q3 + 3 * IQR)]

results = []
tcga_folders = [f for f in os.listdir(input_folder) if os.path.isdir(os.path.join(input_folder, f)) and f.startswith("TCGA")]

for slide_name in tcga_folders:
    file_path = os.path.join(input_folder, slide_name, "features", f"{slide_name}.csv")
    if not os.path.exists(file_path):
        continue

    df = pd.read_csv(file_path)
    df[["Area", "Major Axis", "Minor Axis", "Perimeter"]] *= MPP  

    df = remove_outliers(df, "Major Axis")
    df = remove_outliers(df, "Minor Axis")

    tile_nucleus_counts = df.groupby("Tile")["Nucleus ID"].count().reset_index()
    tile_nucleus_counts.rename(columns={"Nucleus ID": "Nucleus_Count"}, inplace=True)

    threshold_value = tile_nucleus_counts["Nucleus_Count"].quantile(args.percentile / 100)
    top_tiles = tile_nucleus_counts[tile_nucleus_counts["Nucleus_Count"] >= threshold_value]

    df_filtered = df[df["Tile"].isin(top_tiles["Tile"])]
    if df_filtered.empty:
        continue

    mean_values = df_filtered[columns_to_compute].mean()
    std_values = df_filtered[columns_to_compute].std()
    results.append([slide_name, len(tile_nucleus_counts), len(top_tiles)] + mean_values.tolist() + std_values.tolist())

result_df = pd.DataFrame(results, columns=["Slide_Name", "Total_Tiles", "Filtered_Tiles"] +
                          [f"Mean {col}" for col in columns_to_compute] + 
                          [f"Std {col}" for col in columns_to_compute])

result_df.to_csv(output_file_path, index=False)
print(f"Filtered results saved to: {output_file_path}")


Working directory: /data/Lab_ruppin/Ranjan/HnE/
Slide: TCGA-D8-A13Z-01Z-00-DX1_3624_tiles
  - Total tiles with cancer nuclei: 3067
  - 75th percentile nucleus count threshold: 104.0
  - Number of tiles selected (top 25%): 770
Slide: TCGA-AR-A0TR-01Z-00-DX1_3099_tiles
  - Total tiles with cancer nuclei: 2878
  - 75th percentile nucleus count threshold: 223.0
  - Number of tiles selected (top 25%): 721
Slide: TCGA-D8-A1JN-01Z-00-DX1_4957_tiles
  - Total tiles with cancer nuclei: 4951
  - 75th percentile nucleus count threshold: 91.0
  - Number of tiles selected (top 25%): 1245
Slide: TCGA-A2-A0ES-01Z-00-DX1_4815_tiles
  - Total tiles with cancer nuclei: 3895
  - 75th percentile nucleus count threshold: 124.0
  - Number of tiles selected (top 25%): 976
Slide: TCGA-C8-A12U-01Z-00-DX1_4200_tiles
  - Total tiles with cancer nuclei: 3915
  - 75th percentile nucleus count threshold: 167.0
  - Number of tiles selected (top 25%): 990
Slide: TCGA-D8-A1JU-01Z-00-DX1_1890_tiles
  - Total tiles with

In [2]:
# result_df

,Slide_Name,Total_Tiles,Filtered_Tiles,Mean Area,Mean Major Axis,Mean Minor Axis,Mean Perimeter,Mean Eccentricity,Mean Circularity,Std Area,Std Major Axis,Std Minor Axis,Std Perimeter,Std Eccentricity,Std Circularity
0,TCGA-D8-A13Z-01Z-00-DX1_3624_tiles,3067,770,15.418755,5.614750,3.623911,15.739790,0.715026,0.726789,8.738497,1.736600,1.114784,4.556289,0.150470,0.100089
1,TCGA-AR-A0TR-01Z-00-DX1_3099_tiles,2878,721,11.008577,4.714941,3.153885,13.346091,0.698523,0.737001,5.874191,1.305372,0.888298,3.605306,0.149508,0.099834
2,TCGA-D8-A1JN-01Z-00-DX1_4957_tiles,4951,1245,10.014065,4.323268,3.125443,12.472554,0.646506,0.784754,3.793265,0.929303,0.656990,2.484995,0.151348,0.086261
3,TCGA-A2-A0ES-01Z-00-DX1_4815_tiles,3895,976,11.380890,4.583000,3.342480,13.223526,0.639347,0.789252,4.738762,1.078397,0.716613,2.875026,0.148913,0.083765
4,TCGA-C8-A12U-01Z-00-DX1_4200_tiles,3915,990,13.390488,5.040054,3.525133,14.524594,0.669280,0.747041,7.155830,1.377498,1.057996,3.937260,0.158527,0.113454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1101,TCGA-BH-A18S-01Z-00-DX1_1472_tiles,1391,351,10.075491,4.397262,3.114423,12.649830,0.656577,0.771381,4.109242,1.092722,0.627684,2.843544,0.149277,0.090938
1102,TCGA-OK-A5Q2-01Z-00-DX2_4517_tiles,4080,1036,9.115289,4.306793,2.913828,12.293953,0.688415,0.738391,4.193151,1.136062,0.693785,3.105289,0.149459,0.106288
1103,TCGA-EW-A1IX-01Z-00-DX1_2878_tiles,2503,639,7.932001,4.047490,2.691917,11.433014,0.692919,0.748016,4.039276,1.211220,0.654713,3.295550,0.151148,0.114692
1104,TCGA-B6-A0IO-01Z-00-DX1_2044_tiles,1868,467,11.182033,4.835227,3.153959,13.645222,0.710739,0.722369,5.692842,1.341342,0.866114,3.620453,0.149988,0.109257


In [3]:
# result_df.describe()

,Total_Tiles,Filtered_Tiles,Mean Area,Mean Major Axis,Mean Minor Axis,Mean Perimeter,Mean Eccentricity,Mean Circularity,Std Area,Std Major Axis,Std Minor Axis,Std Perimeter,Std Eccentricity,Std Circularity
count,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000,1106.000000
mean,2385.485533,602.500000,13.492241,5.015020,3.431469,14.288192,0.679219,0.750655,7.110009,1.405320,0.926723,3.803815,0.151660,0.100489
std,1315.462462,331.788318,7.544913,1.154716,0.745445,3.222503,0.023135,0.023725,5.004859,0.462752,0.334439,1.296602,0.003667,0.010479
min,62.000000,17.000000,5.459585,3.448275,2.273064,9.892838,0.592410,0.570934,2.226721,0.712878,0.384966,1.897839,0.141263,0.071850
25%,1391.000000,353.250000,9.990349,4.423162,3.048560,12.620209,0.665191,0.738082,4.586128,1.136760,0.729691,3.050981,0.149399,0.093926
50%,2242.500000,566.500000,11.413260,4.710690,3.233216,13.456250,0.681491,0.752461,5.704438,1.290685,0.841153,3.488905,0.151345,0.099128
75%,3164.750000,799.750000,13.217866,5.075274,3.468552,14.444155,0.694737,0.765610,7.213625,1.486319,0.989287,4.048707,0.153677,0.105577
max,9153.000000,2289.000000,53.595802,10.112775,6.962494,28.829157,0.756216,0.813011,38.947349,3.878163,2.819529,10.994694,0.172392,0.155437
